In [1]:
import os
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from pyspark.sql import DataFrame
from etl.transformations.common import (
    create_spark,
    read_clickhouse,
    write_clickhouse,
)
from etl.transformations.facts.common import (
    add_date_key,
    build_daily_sensor_metrics,
)

In [2]:
def transform_daily_sensor_metrics(
    fact_sensor_readings_df: DataFrame,
    dim_date_df: DataFrame,
) -> DataFrame:
    """
    Build daily metrics per farm and sensor type.
    """

    metrics_df = build_daily_sensor_metrics(
        fact_sensor_readings_df,
    )

    metrics_df = add_date_key(
        metrics_df,
        dim_date_df,
        "metric_date",
    )

    return metrics_df.select(
        "metric_date",
        "date_key",
        "farm_key",
        "farm_id",
        "sensor_type_key",
        "reading_count",
        "sum_value",
        "min_value",
        "max_value",
        "anomaly_count",
        "in_range_count",
    )


def main():
    spark = create_spark(
        "fact_daily_sensor_metrics",
    )

    try:
        fact_sensor_readings_df = read_clickhouse(
            spark,
            "fact_sensor_readings",
        )

        dim_date_df = read_clickhouse(
            spark,
            "dim_date",
        )

        daily_sensor_metrics_df = transform_daily_sensor_metrics(
            fact_sensor_readings_df,
            dim_date_df,
        )

        write_clickhouse(
            daily_sensor_metrics_df,
            "fact_daily_sensor_metrics",
        )

    finally:
        spark.stop()


if __name__ == "__main__":
    main()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/23 10:14:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/23 10:15:00 WARN JdbcUtils: Requested isolation level 1, but transactions are unsupported
26/07/23 10:15:00 WARN JdbcUtils: Requested isolation level 1, but transactions are unsupported
26/07/23 10:15:00 WARN JdbcUtils: Requested isolation level 1, but transactions are unsupported
26/07/23 10:15:00 WARN JdbcUtils: Requested isolation level 1, but transactions are unsupported
26/07/23 10:15:00 WARN JdbcUtils: Requested isolation level 1, but transactions are unsupported
26/07/23 10:15:00 WARN JdbcUtils: Requested isolation level 1, but transactions are unsupported
